In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from django_pandas.io import read_frame
from intecomm_rando.models import RandomizationList
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_rando.constants import COMMUNITY_ARM, FACILITY_ARM
from intecomm_analytics.constants import HIV_ALONE, DM_ALONE, HTN_ALONE, HTN_DM

In [ ]:
df_rando = read_frame(RandomizationList.objects.values("group_identifier", "assignment").filter(group_identifier__isnull=False))

In [ ]:
df_main = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df1 = df_main.copy()

In [ ]:
df_group_comm = df1[df1.assignment==COMMUNITY_ARM].groupby(by=["group_identifier"]).size().to_frame()
df_group_comm = df_group_comm.reset_index()
df_group_comm.columns = ["group_identifier", "group_size"]
df_group_comm["group_size"].sum()
df_group_comm["group_size"].describe()

In [ ]:
df_group_facility = df1[df1.assignment==FACILITY_ARM].groupby(by=["group_identifier"]).size().to_frame()
df_group_facility = df_group_facility.reset_index()
df_group_facility.columns = ["group_identifier", "group_size"]
df_group_facility["group_size"].sum()
df_group_facility["group_size"].describe()

In [ ]:
df1[df1.assignment==COMMUNITY_ARM].gender.value_counts()

In [ ]:
df1[df1.assignment==FACILITY_ARM].gender.value_counts()

In [ ]:

df2 = df1[df1.primary_cohort.isin([HIV_ALONE, DM_ALONE, HTN_ALONE, HTN_DM])].groupby(by=["assignment", "primary_cohort"]).size().to_frame().reset_index()
df2["primary_cohort"] = df2.primary_cohort.apply(lambda x: 5 if x <4 else x)
df2[(df2.primary_cohort==5) & (df2.assignment==FACILITY_ARM)].sum()


In [ ]:
df2 = df_main.copy()
df2.loc[(df2.hiv==1) & ((df2.dm==1) | (df2.htn==1)), "hiv_ncd"] = 1
df_group_cond = df2.groupby(by=["group_identifier"]).agg({"ncd":"sum", "hiv_only":"sum", "hiv_ncd":"sum"})
df_group_cond["group_size_check"] = df_group_cond["ncd"] + df_group_cond["hiv_only"] + df_group_cond["hiv_ncd"]
df_group_cond

In [ ]:
df_group_size = df2.groupby(by=["group_identifier"]).size().to_frame().reset_index()
df_group_size.columns = ["group_identifier", "group_size"]
df_group_size

In [ ]:
df_group = pd.merge(df_group_cond, df_group_size, on="group_identifier", how="left")

In [ ]:
# group size caclulated from conditions (group_size_check) should match group size (group_size)
df_group[df_group.group_size_check != df_group.group_size]

In [ ]:
df_group.drop(columns=["group_size_check"], inplace=True)

In [ ]:
df_group["ncd_perc"] = df_group["ncd"] / df_group["group_size"]
df_group["hiv_only_perc"] = df_group["hiv_only"] / df_group["group_size"]
df_group["hiv_ncd_perc"] = df_group["hiv_ncd"] / df_group["group_size"]
df_group["tot_perc"] = df_group["ncd_perc"] +  df_group["hiv_only_perc"] + df_group["hiv_ncd_perc"]
df_group

In [ ]:
df_group = df_group.merge(df_rando, on="group_identifier", how="left")
df_group.reset_index(drop=True, inplace=True)
df_group

In [ ]:
df_group.ncd_perc.describe()

In [ ]:
df_group.hiv_only_perc.describe()
dfx = df_group.hiv_only_perc.describe().to_frame()
dfx = dfx.reset_index()
dfx.columns=["statistic", "value"]
dfx = dfx.pivot_table(columns=["statistic"], values="value")
dfx = dfx.reset_index(drop=True)
dfx["index"] = 0
dfx.set_index("index", inplace=True)
dfx.index.name = "index"
# dfx.drop(columns=["statistic"], inplace=True)
dfx["label"] = "hiv_only"
# dfx.drop(columns=["statistic"], inplace=True)
dfx_hiv_only = dfx.copy()

In [ ]:
dfx = df_group.hiv_ncd_perc.describe().to_frame()
dfx = dfx.reset_index()
dfx.columns=["statistic", "value"]
dfx = dfx.pivot_table(columns=["statistic"], values="value")
dfx = dfx.reset_index(drop=True)
dfx["index"] = 0
dfx.set_index("index", inplace=True)
dfx.index.name = "index"
# dfx.drop(columns=["statistic"], inplace=True)
dfx["label"] = "hiv_ncd"
# dfx.drop(columns=["statistic"], inplace=True)
dfx_hiv_ncd = dfx.copy()

In [ ]:
dfx = df_group.ncd_perc.describe().to_frame()
dfx = dfx.reset_index()
dfx.columns=["statistic", "value"]
dfx = dfx.pivot_table(columns=["statistic"], values="value")
dfx = dfx.reset_index(drop=True)
dfx["index"] = 0
dfx.set_index("index", inplace=True)
dfx.index.name = "index"
# dfx.drop(columns=["statistic"], inplace=True)
dfx["label"] = "ncd"
# dfx.drop(columns=["statistic"], inplace=True)
dfx_ncd = dfx.copy()

In [ ]:
# overall by condition
df_group_stats = pd.concat([dfx_ncd, dfx_hiv_ncd, dfx_hiv_only])
df_group_stats

In [ ]:
df_group_size = df1.groupby(by=["group_identifier"]).size().to_frame().reset_index()
df_group_size.columns = ["group_identifier", "group_size"]
df_group_size["group_size"].sum()

In [ ]:
df_group_size["group_size"].describe()

In [ ]:
df_group_stats = pd.concat([dfx_ncd, dfx_hiv_ncd, dfx_hiv_only])
df_group_stats

In [ ]:
df_group_stats["min"] = df_group_stats["min"] * 11
df_group_stats["max"] = df_group_stats["max"] * 20
df_group_stats["mean"] = df_group_stats["mean"] * 14.983871
df_group_stats["25%"] = df_group_stats["25%"] * 14
df_group_stats["50%"] = df_group_stats["50%"] * 15
df_group_stats["75%"] = df_group_stats["75%"] * 16
df_group_stats["std"] = df_group_stats["std"] * 1.748327

In [ ]:
df_group_stats = df_group_stats.apply(round)

In [ ]:
df_group_stats

mean ratio of 10:4:1 (NCD:HIV:HIV_NCD) (5-17:0-9:0-6)